# M7 — Version E: the agent (LangGraph + MCP tools + memory)

**Goal of this step:** instead of a fixed pipeline (A, B, D always do the same steps), build an **agent** that
*decides for itself* which tools to use, looks at the results, fixes its own mistakes, and remembers.

**Analogy:** versions A–D are like a vending machine: same steps every time. The agent is like an analyst who
searches the catalog, checks *"is this the hourly or the daily table?"*, runs a query, sees an error, fixes it,
and remembers what you asked a minute ago.

**ReAct = REason + ACT, in a loop**
```
START → agent (think: which tool next?) → tools (run them through MCP) → agent (look at results) → … → final answer
```

**The agent's rules (its "harness")**
| Rule | Why |
|---|---|
| at most **8 steps** per question | stops endless loops |
| at most **3 failed SQL runs** | stops endless retrying |
| a **token budget** per question (60k) | a cost limit |
| when a limit is hit: **best answer now, or "not sure"** | never guess silently |

**Two kinds of memory**
| Memory | What it remembers | How |
|---|---|---|
| **Chat memory** (short-term) | earlier turns of *this* conversation, so "now split **that** by age" works | LangGraph *checkpointer*, one *thread* per conversation; only the last 6 user turns are sent to the AI |
| **Example memory** (long-term) | practice questions it answered **correctly** (checked against the gold answer) | saved to `memory/examples.jsonl`, recalled by meaning (like M2) |

**Files in this step**
| File | What it does |
|---|---|
| `agent/mcp_tools.py` | connects to the MCP server (M6) and turns its 6 tools into tools the agent can call |
| `agent/agent_graph.py` | the LangGraph agent: the loop, the rules, the chat memory |
| `agent/memory.py` | example memory: `remember()` and `recall()` |
| `agent/pipelines.py` | adds `version_e()` |
| `harness/run.py` | now also runs **conversations** (`--conversations`) and fills memory from practice (`--learn`) |
| `configs/E.yaml`, `E_memory.yaml`, `E_nochat.yaml` | E with different memory switched on/off |

**Done when:** follow-up conversations work, and the limits stop runaway loops ✅

In [1]:
import sys, json, glob
for folder in ["../agent", "../catalog", "../harness", "../mcp_server"]:
    sys.path.append(folder)

## 1. Start the agent
`AgentE` starts the MCP server (a separate program), asks it for its tools, and builds the LangGraph graph.
The agent never imports the data code: everything goes through MCP.

In [2]:
from agent_graph import AgentE
agent = AgentE({"model": "gpt-5-mini", "reasoning_effort": "low", "chat_memory": True, "example_memory": False})
print("tools from MCP:", [t["name"] if isinstance(t, dict) else t.name for t in agent.connection.list_tools()])

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tools from MCP: ['search_tables', 'get_join_path', 'list_lookalikes', 'describe_table', 'sample_rows', 'run_sql']


The graph itself: two nodes (agent, tools) and a loop between them.

In [3]:
print(agent.agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Watch it think: one question, step by step
A tricky one: "excluding suspicious clicks" means the **valid** look-alike table.

In [4]:
r = agent.ask("What was campaign 143747's CTR yesterday, excluding suspicious clicks?", thread_id="demo-1")
print(f"steps: {r['steps']}   failed SQL: {r['sql_errors']}   tokens in/out: {r['tokens_in']}/{r['tokens_out']}\n")
for call in r["tool_calls"]:
    print("→", call)
print("\nANSWER:\n" + r["answer"])

steps: 4   failed SQL: 0   tokens in/out: 4509/280

→ search_tables({"question": "campaign CTR yesterday excluding suspicious clicks campaign_id 143747 suspicious clicks", "k": 5})
→ describe_table({"table": "agg_campaign_daily_valid"})
→ run_sql({"sql": "SELECT event_date, campaign_id, impressions, clicks, ctr\nFROM agg_campaign_daily_valid\nWHERE campaign_id = 14)

ANSWER:
Campaign 143747's CTR for yesterday (2017-05-12), excluding suspicious clicks, was 0.056338028169014086 (clicks/impressions).

SQL used:
```sql
SELECT event_date, campaign_id, impressions, clicks, ctr
FROM agg_campaign_daily_valid
WHERE campaign_id = 143747
  AND event_date = DATE '2017-05-12';
```


## 3. Chat memory: a follow-up in the same conversation
Same `thread_id` = same conversation. The question doesn't say which campaign or which day; the agent has to
remember.

In [5]:
r = agent.ask("And how many unique people saw it that day?", thread_id="demo-1")
for call in r["tool_calls"]:
    print("→", call)
print("\nANSWER:\n" + r["answer"][:800])

→ search_tables({"question": "unique people per campaign daily valid unique people saw campaign daily 2017-05-12", "k": 5})
→ describe_table({"table": "reach_campaign_daily"})
→ run_sql({"sql": "SELECT event_date, campaign_id, unique_users\nFROM reach_campaign_daily\nWHERE campaign_id = 143747\n  AND even)

ANSWER:
141 unique people saw campaign 143747 on 2017-05-12.

Note: reach_campaign_daily counts unique users (people) regardless of whether clicks were later marked suspicious. 

SQL used:
```sql
SELECT event_date, campaign_id, unique_users
FROM reach_campaign_daily
WHERE campaign_id = 143747
  AND event_date = DATE '2017-05-12';
```


**Without** chat memory (a new thread), the same follow-up has nothing to refer to:

In [6]:
r = agent.ask("And how many unique people saw it that day?", thread_id="demo-new-thread")
print(r["answer"][:600])

I don't know which "it" or which day you mean. Tell me the entity type and ID (ad_id, campaign_id, brand_id, category_id, or placement_id) and the date (YYYY‑MM‑DD). The data covers 2017-05-06 through 2017-05-13.

While you tell me, here are ready-to-run SQL templates you can use. "Unique people" = count distinct user_id in fact_ad_events.

- For one ad on one day (replace :AD_ID and :DAY):
```sql
SELECT COUNT(DISTINCT user_id) AS unique_people
FROM fact_ad_events
WHERE date = DATE '2017-05-13'   -- replace with :DAY
  AND ad_id = 12345;             -- replace with :AD_ID
```

- For one campai


## 4. Example memory (long-term)
`harness/run.py --learn` ran E on the 40 **practice** questions and saved each one it answered **correctly**.
`recall()` finds the most similar saved examples for a new question. Test questions can never be saved (the
harness refuses `--learn` on the test split).

In [7]:
from memory import load_examples, recall
examples = load_examples()
print(f"{len(examples)} examples in memory, e.g.:\n  {examples[0]['question']}\n  {examples[0]['sql']}\n")
for e in recall("Campaign 12345: unique users reached in the last six hours?"):
    print("recalled:", e["question"])

34 examples in memory, e.g.:
  Give me the click-through rate for campaign 9865 on 11 May 2017.
  SELECT event_date, campaign_id, impressions, clicks, ctr
FROM agg_campaign_daily
WHERE campaign_id = 9865
  AND event_date = DATE '2017-05-11';



recalled: Campaign 333637: unique users reached in the last six hours?
recalled: Campaign 81421: unique users reached in the last six hours?


In [8]:
agent.close()

## 5. Results on the practice questions: A vs B vs D vs E
```
.venv/bin/python harness/run.py configs/E.yaml --split practice --learn
```

In [9]:
def latest(name, suffix="practice"):
    folders = sorted(glob.glob(f"../results/*_{name}_{suffix}"))
    return (folders[-1], json.load(open(folders[-1] + "/metrics.json"))) if folders else (None, None)

runs = {v: latest(v) for v in ["A", "B", "D", "E"]}
keys = ["accuracy", "accuracy_easy", "accuracy_medium", "accuracy_tricky", "table_f1", "gold_tables_shown",
        "not_sure", "avg_steps", "avg_seconds", "avg_tokens_in", "avg_cost_usd"]
print(f"{'metric':18s}" + "".join(f"{v:>9s}" for v in runs))
for k in keys:
    print(f"{k:18s}" + "".join(f"{runs[v][1].get(k, '-'):>9}" for v in runs))

metric                    A        B        D        E
accuracy              0.875      0.7     0.85     0.85
accuracy_easy           1.0      1.0      1.0   0.8462
accuracy_medium      0.8462   0.3077   0.6923   0.7692
accuracy_tricky      0.7857   0.7857   0.8571   0.9286
table_f1             0.9542   0.7208   0.8292   0.8167
gold_tables_shown       1.0     0.65     0.85    0.825
not_sure                  -        -        -        0
avg_steps                 -        -        -     5.05
avg_seconds          3.7472   4.2538   3.7412   9.7327
avg_tokens_in        3208.8  712.825  911.225 6730.725
avg_cost_usd         0.0012   0.0007   0.0007   0.0025


## 6. Follow-up conversations: chat memory on vs off
5 practice conversations × 3 turns. Turn 1 is a normal question; turns 2 and 3 only make sense with memory.
```
.venv/bin/python harness/run.py configs/E.yaml --split practice --conversations
.venv/bin/python harness/run.py configs/E_nochat.yaml --split practice --conversations
```

In [10]:
conv = {name: latest(name, "practice_conv") for name in ["E", "E_nochat"]}
print(f"{'metric':16s}{'memory ON':>12s}{'memory OFF':>12s}")
for k in ["accuracy", "accuracy_turn1", "accuracy_turn2", "accuracy_turn3", "avg_cost_usd"]:
    print(f"{k:16s}" + "".join(f"{conv[n][1].get(k, '-'):>12}" for n in conv))

metric             memory ON  memory OFF
accuracy                 0.6      0.2667
accuracy_turn1           0.6         0.8
accuracy_turn2           0.6         0.0
accuracy_turn3           0.6         0.0
avg_cost_usd          0.0019      0.0016


## 7. E's mistakes

In [11]:
for line in open(runs["E"][0] + "/answers.jsonl"):
    r = json.loads(line)
    if not r["correct"]:
        print(f"{r['id']} {r['template']:3s} steps={r['steps']} {r['question'][:70]}")
        print(f"      used: {r['tables_used']}  gold: {r['gold_tables']}  error: {r['error'][:60]}")

q001 E1  steps=4 What was the CTR of campaign 279732 on 6 May 2017?
      used: ['agg_campaign_daily_valid']  gold: ['agg_campaign_daily']  error: 
q035 E9  steps=4 Which ad placement had the higher CTR on 6 May 2017?
      used: ['agg_placement_daily']  gold: ['agg_placement_daily']  error: 
q054 M4  steps=8 Estimated value per category for advertiser 224993 yesterday?
      used: ['dim_ad', 'fact_ad_events']  gold: ['agg_ad_daily', 'dim_ad']  error: 
q055 M4  steps=5 Break down advertiser 141337's estimated value yesterday by product ca
      used: ['fact_ad_events']  gold: ['agg_ad_daily', 'dim_ad']  error: 
q056 M4  steps=6 Estimated value per category for advertiser 183758 yesterday?
      used: ['fact_ad_events']  gold: ['agg_ad_daily', 'dim_ad']  error: 
q101 T6  steps=4 How many clicks did campaign 319897 get in the week starting Monday 8 
      used: ['agg_campaign_weekly']  gold: ['agg_campaign_weekly']  error: 


## 8. What this tells us
Read the numbers above together with these questions. The final README text will be written from the
**test** results in M9.
- **Does acting step by step help on tricky (look-alike) questions?** The agent can call `list_lookalikes` and
  `describe_table` before choosing a table.
- **What does it cost?** The agent makes several AI calls per question, so it uses more tokens and time than A–D.
  Is the accuracy gain worth it?
- **Chat memory** should make turns 2 and 3 work. Without it, "split **that**" has nothing to refer to.
- **Example memory** is tested on the **test** questions in M9 (E vs E_memory). It would be unfair on practice,
  because the memory was filled *from* those same practice questions.

**Next step (M9, final runs):** run A, B, D, E and E_memory on the 80 test questions (3 runs each) plus the 15
test conversations, then confidence intervals and the failure analysis for the README.